# siRNA Dataset

### Import Libraries

In [2]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt # data visualization
import seaborn as sns # statistical data visualization
import sklearn as sk

### Import Dataset

In [30]:
data = 'siRNA/data.csv'

df = pd.read_csv(data)

### Exploratory Data Analysis

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
#Columns names

col_names = df.columns

print(col_names)

In [ ]:
df.info()

In [ ]:
#Checking missing values

null_values = df.isnull().sum()
print(null_values)

In [ ]:
df['Target seq'].value_counts()

In [ ]:
# Excluding columns that include ID num (to be ıncluded only int.)
df = df.drop(['Target seq', 'Sequence'], axis=1)

X = df.iloc[:, :-1]
y = df['Activity'] 

# Display first few rows
print(X.head())
print(y.head())

# Summary statistics
print(X.describe())
print(y.describe())

## Black Box Algorithms

#### Import Libraries

* Requirements: **tensorflow**

In [32]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import root_mean_squared_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

### Deep Neural Network

#### Feature Scaling

In [12]:
# Initialize the scaler
#scaler = StandardScaler()

# Fit the scaler on the entire dataset and transform
#X_scaled = scaler.fit_transform(X)

#### Split Dataset into Test and Train

In [ ]:
# Split the dataset into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples: {X_test.shape[0]}")

#### Model 1

In [ ]:
# Define the model architecture
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='linear')  # Output layer for regression
])

# Display the model's architecture
model.summary()

In [54]:
model.compile(
    optimizer='adam',
    loss='mean_squared_error',
    metrics=['mean_squared_error']
)

In [ ]:
# Define Early Stopping callback
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=20,
    restore_best_weights=True
)

# Train the model
# training set will be splitted in validation set 20%
history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# Evaluate on test data
test_loss, test_mse = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Mean Squared Error (MSE): {test_mse:.4f}")
print(f"Test Loss: {test_loss:.4f}")

# Predict on test data
y_pred = model.predict(X_test).flatten()

# Calculate additional metrics
rmse = np.sqrt(root_mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"Test Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"Test R-squared (R²): {r2:.4f}")

In [ ]:
plt.figure(figsize=(10,6))
plt.plot(history.history['mean_squared_error'], label='Training MSE')
plt.plot(history.history['val_mean_squared_error'], label='Validation MSE')
plt.xlabel('Epoch')
plt.ylabel('Mean Squared Error')
plt.title('Training and Validation MSE Over Epochs')
plt.legend()
plt.show()

#### Model 2

In [ ]:
model2 = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.2),
    Dense(128, activation='relu'),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='linear')
])

model2.summary()

In [ ]:
model2.compile(
    optimizer='adam', 
    loss='mean_squared_error',
    metrics=['mean_squared_error']
    )

early_stop = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)

# Train the model
history = model2.fit(
    X_train, y_train,
    epochs=200,  # Increased epochs to ensure model has time to train
    batch_size=32,
    validation_split=0.3,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# Evaluate on test data
test_loss, test_mse = model2.evaluate(X_test, y_test, verbose=0)
print(f"Test Mean Squared Error (MSE): {test_mse:.4f}")
print(f"Test Loss: {test_loss:.4f}")

# Predict on test data
y_pred = model2.predict(X_test).flatten()

# Calculate additional metrics
rmse = np.sqrt(root_mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"Test Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"Test R-squared (R²): {r2:.4f}")

In [ ]:
plt.figure(figsize=(10,6))
plt.plot(history.history['mean_squared_error'], label='Training MSE')
plt.plot(history.history['val_mean_squared_error'], label='Validation MSE')
plt.xlabel('Epoch')
plt.ylabel('Mean Squared Error')
plt.title('Training and Validation MSE Over Epochs')
plt.legend()
plt.show()

## Decision Tree

### CART

In [22]:
# Initialize the scaler
#scaler = StandardScaler()

# Fit the scaler on the entire dataset and transform
#X_scaled = scaler.fit_transform(X)

#### Split into Train, Test, Validation

In [27]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Split the test data into test and validation
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

In [ ]:
from sklearn.tree import DecisionTreeRegressor

# Initialize the regressor
model_DT = DecisionTreeRegressor(
    criterion='squared_error',  # Default is 'squared_error' for regression
    random_state=42
)

# Train the model
model_DT.fit(X_train, y_train)

#### Evaluation Metrics

In [ ]:
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

# Predict on the validation set
val_pred = model_DT.predict(X_val)

# Calculate evaluation metrics
mse = root_mean_squared_error(y_val, val_pred)
rmse = np.sqrt(mse)  # Root Mean Squared Error
mae = mean_absolute_error(y_val, val_pred)
r2 = r2_score(y_val, val_pred)

# Print the results
print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"R-squared (R²) Score: {r2:.4f}")

In [ ]:
#Predict on the test set
test_pred = model_DT.predict(X_test)

# Calculate evaluation metrics
mse = root_mean_squared_error(y_test, test_pred)
rmse = np.sqrt(mse)  # Root Mean Squared Error
mae = mean_absolute_error(y_test, test_pred)
r2 = r2_score(y_test, test_pred)

# Print the results
print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"R-squared (R²) Score: {r2:.4f}")

#### Visualizing

In [ ]:
# Scatter plot of Actual vs. Predicted
sns.scatterplot(x=y_test, y=test_pred)
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.title('Actual vs. Predicted Values (Test Set)')
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--')  # Diagonal line
plt.show()

plt.scatter(y_val, val_pred)
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.title('Actual vs Predicted (Validation Set)')
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--')  # Diagonal line
plt.show()


### C4.5

In [ ]:
pip install chefboost

In [2]:
from chefboost import Chefboost as chef

In [6]:
df_c4 = pd.read_csv(data)

df_15 = df_c4.copy()
df_20 = df_c4.copy()

In [ ]:
mean_activity = np.mean(df_c4.Activity)
median_activity = np.median(df_c4.Activity)

print(mean_activity)
print(median_activity)

In [8]:
# Example threshold (choose based on your data distribution)
threshold = 15  

# Convert continuous target 'Activity' into binary
df_15['Activity'] = np.where(df_c4.Activity < threshold, 'Yes', 'No')

In [ ]:
df_15.head()

In [ ]:
config = {'algorithm': 'C4.5'} #Set algorithm to C4.5
model_c4_15 = chef.fit(df_15, config, 'Activity')

In [92]:
# Example threshold (choose based on your data distribution)
threshold = 20  

# Convert continuous target 'Activity' into binary
df_20['Activity'] = np.where(df_c4.Activity < threshold, 'Yes', 'No')

In [ ]:
df_20.head()

In [ ]:
config = {'algorithm': 'C4.5'} #Set algorithm to C4.5
model_c4 = chef.fit(df_20, config, 'Activity')